In [1]:
# !pip install wandb

In [2]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from src.preprocess import BasicDegenderizer, AdvancedDegenderizer
from src.models import DistilBERTClassifier, RoBERTaClassifier

/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/__init__.py:64: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

In [3]:
# import wandb

In [4]:
# wandb.login(key="5489aee4351c1c3af108d0f20e5191f366756c2c")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hice1/mwesley32/.netrc
wandb: Currently logged in as: mtwesley to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
# wandb.init(
#     project="NLP-Letters-V2",
#     group="nlp-letters-v2-distilbert-gendered"
# )

In [6]:
DATA_PATH = "data/combined_letters_degendered.csv"
LABEL_COLUMN = "label"
TEXT_COLUMN = "full_text"
DEGENDERIZERS = []

In [7]:
# Load dataset
df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")
print("Dataset shape:", df.shape)

Dataset shape: (3285, 19)


In [8]:
df.head()

,applicant_id,document,writer_gender,letter_type,applicant_gender,applicant_identify,applicant_identify_group,standardized_lor,us_canadian,usmle_1,usmle_2,applicants_writers,TEXT,full_text,s1,s2,s1_s2,s3,full_text_tokens
0,male_3090,sentence_sets.csv.4358,female,lor,male,white,white,1,1,197,225,male_female,this letter serves as the department of medici...,this letter serves as the department of medici...,FIRST_NAME LAST_NAME * LAST_NAME waived his r...,this letter serves as the department of medici...,FIRST_NAME LAST_NAME * LAST_NAME waived his r...,these letters are written and compiled by the ...,526
1,female_267,sentence_sets.csv.4301,male,lor,female,asian - chinese,asian,1,1,203,227,female_male,this letter is written in support of FIRST_NAM...,this letter is written in support of FIRST_NAM...,this letter is written in support of FIRST_NAM...,she is reliable responsible and well reg...,this letter is written in support of FIRST_NAM...,our residents rotate in both inpatient and amb...,447
2,male_1004,sentence_sets.csv.179,male,lor,male,"hispanic, latino, or of spanish origin - mexic...",hispanic_latino_spanish,1,1,206,237,male_male,comments from anesthesiology rotations : â F...,comments from anesthesiology rotations FIRS...,comments from anesthesiology rotations FIRS...,he had excellent questions to showed his since...,comments from anesthesiology rotations FIRS..., *  *  * works hard always looking for w...,365
3,female_147,sentence_sets.csv.3846,male,lor,female,"hispanic, latino, or of spanish origin - other...",hispanic_latino_spanish,1,1,206,217,female_male,long number ms LAST_NAME had originally planne...,long number ms LAST_NAME had originally planne...,long number ms LAST_NAME had originally planne...,she took full advantage of the opportunity f...,long number ms LAST_NAME had originally planne...,2 I am not aware of any areas that require...,346
4,male_381,sentence_sets.csv.6,female,lor,male,black or african american - african american|b...,black or african american,1,1,210,225,male_female,â i think [ FIRST_NAME ] would be an asset t...,i think [ FIRST_NAME ] would be an asset to a...,i think [ FIRST_NAME ] would be an asset to an...,he's someone I look forward to speaking with a...,i think [ FIRST_NAME ] would be an asset to an...,* ucsd would be fortunate to get to keep this...,512


In [9]:
# advanced_pipeline = Pipeline([("advanced", AdvancedDegenderizer(paths=DEGENDERIZERS))])
# df["degendered"] = advanced_pipeline.fit_transform(df[TEXT_COLUMN].tolist())
# df["gendered"] = df[TEXT_COLUMN]

In [4]:
# # Create degendering pipelines for each degenderizer
# basic_pipeline = Pipeline([("basic", BasicDegenderizer(paths=DEGENDERIZERS))])
# advanced_pipeline = Pipeline([("advanced", AdvancedDegenderizer(paths=DEGENDERIZERS))])

In [27]:
# # Apply both pipelines to the text column
# df["basic_degendered"] = basic_pipeline.fit_transform(df[TEXT_COLUMN].tolist())
# df["advanced_degendered"] = advanced_pipeline.fit_transform(df[TEXT_COLUMN].tolist())

# print(df[[TEXT_COLUMN, "gendered", "degendered"]].head())

                                           full_text  \
0  this letter serves as the department of medici...   
1  this letter is written in support of FIRST_NAM...   
2  comments from anesthesiology rotations    FIRS...   
3  long number ms LAST_NAME had originally planne...   
4   i think [ FIRST_NAME ] would be an asset to a...   

                                            gendered  \
0  this letter serves as the department of medici...   
1  this letter is written in support of FIRST_NAM...   
2  comments from anesthesiology rotations    FIRS...   
3  long number ms LAST_NAME had originally planne...   
4   i think [ FIRST_NAME ] would be an asset to a...   

                                          degendered  
0  this letter serves as the department of medici...  
1  this letter is written in support of FIRST_NAM...  
2  comments from anesthesiology rotations    FIRS...  
3  long number ms LAST_NAME had originally planne...  
4   i think [ FIRST_NAME ] would be an asset to a..

In [6]:
# For training, we use advanced
# df["degendered_text"] = df["advanced_degendered"]

In [10]:
# Convert categorical labels to factors / integers
df["label"], class_mapping = pd.factorize(df["label"])
print("Class mapping:", dict(enumerate(class_mapping)))

Class mapping: {0: 'male', 1: 'female'}


In [11]:
# Train test splits
X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

print("Train size:", len(X_train), "Test size:", len(X_test))

Train size: 2628 Test size: 657


In [12]:
# DistilBERT classifier model
model = DistilBERTClassifier(
    model_name="distilbert-base-uncased",
    num_labels=len(class_mapping),
)

model.train(
    X_train.tolist(),
    y_train.tolist(),
    epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to=["wandb"]
)

metrics = model.test(X_test.tolist()[0:5], y_test.tolist()[0:5])

print("Evaluation Metrics:", metrics)

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/home/hice1/mwesley32/NLP-Letters-V2/src/models.py:189: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Auc Roc,Auc Pr,0 Precision,0 Recall,0 F1,1 Precision,1 Recall,1 F1,Cm 00,Cm 01,Cm 10,Cm 11,Runtime,Samples Per Second,Steps Per Second
1,0.003300,0.002802,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,374,0,0,152,3.037700,173.160000,10.864000
2,0.001500,0.001068,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,374,0,0,152,3.043600,172.820000,10.842000
3,0.001000,0.000832,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,374,0,0,152,3.048300,172.557000,10.826000


Parameter 'function'=<bound method DistilBERTClassifier._tokenize of <src.models.DistilBERTClassifier object at 0x155450bd4730>> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:708: RuntimeWarning: invalid value encountered in true_divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/l

Evaluation Metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'mcc': 0.0, 'balanced_accuracy': 1.0, 'cohen_kappa': nan, 'jaccard': 1.0, 'hamming_loss': 0.0, 'auc_roc': None, 'auc_pr': None, '0_precision': 1.0, '0_recall': 1.0, '0_f1': 1.0, 'cm_00': 5}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:708: RuntimeWarning: invalid value encountered in true_divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/l

In [11]:
model = RoBERTaClassifier(
    model_name="roberta-base",
    num_labels=len(class_mapping),
)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
model.train(
    X_train.tolist(),
    y_train.tolist(),
    epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to=["wandb"]
)

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/home/hice1/mwesley32/NLP-Letters-V2/src/models.py:394: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Auc Roc,Auc Pr,Confusion Matrix,Classification Report,Runtime,Samples Per Second,Steps Per Second
1,No log,0.047470,0.992395,0.992677,0.988795,0.990711,0.981464,0.988795,0.981422,0.981623,0.007605,0.988795,0.979431,"[[373, 1], [3, 149]]","{'0': {'precision': 0.9920212765957447, 'recall': 0.9973262032085561, 'f1-score': 0.9946666666666667, 'support': 374.0}, '1': {'precision': 0.9933333333333333, 'recall': 0.9802631578947368, 'f1-score': 0.9867549668874173, 'support': 152.0}, 'accuracy': 0.9923954372623575, 'macro avg': {'precision': 0.992677304964539, 'recall': 0.9887946805516465, 'f1-score': 0.990710816777042, 'support': 526.0}, 'weighted avg': {'precision': 0.9924004260712455, 'recall': 0.9923954372623575, 'f1-score': 0.9923803960080243, 'support': 526.0}}",5.590200,94.094000,5.903000
2,No log,0.044123,0.992395,0.994709,0.986842,0.990674,0.981520,0.986842,0.981349,0.981551,0.007605,0.986842,0.981289,"[[374, 0], [4, 148]]","{'0': {'precision': 0.9894179894179894, 'recall': 1.0, 'f1-score': 0.9946808510638298, 'support': 374.0}, '1': {'precision': 1.0, 'recall': 0.9736842105263158, 'f1-score': 0.9866666666666667, 'support': 152.0}, 'accuracy': 0.9923954372623575, 'macro avg': {'precision': 0.9947089947089947, 'recall': 0.986842105263158, 'f1-score': 0.9906737588652482, 'support': 526.0}, 'weighted avg': {'precision': 0.9924759088257188, 'recall': 0.9923954372623575, 'f1-score': 0.9923649650783378, 'support': 526.0}}",5.578800,94.285000,5.915000
3,No log,0.043542,0.992395,0.994709,0.986842,0.990674,0.981520,0.986842,0.981349,0.981551,0.007605,0.986842,0.981289,"[[374, 0], [4, 148]]","{'0': {'precision': 0.9894179894179894, 'recall': 1.0, 'f1-score': 0.9946808510638298, 'support': 374.0}, '1': {'precision': 1.0, 'recall': 0.9736842105263158, 'f1-score': 0.9866666666666667, 'support': 152.0}, 'accuracy': 0.9923954372623575, 'macro avg': {'precision': 0.9947089947089947, 'recall': 0.986842105263158, 'f1-score': 0.9906737588652482, 'support': 526.0}, 'weighted avg': {'precision': 0.9924759088257188, 'recall': 0.9923954372623575, 'f1-score': 0.9923649650783378, 'support': 526.0}}",5.549900,94.777000,5.946000


Trainer is attempting to log a value of "[[373, 1], [3, 149]]" of type <class 'list'> for key "eval/confusion_matrix" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'0': {'precision': 0.9920212765957447, 'recall': 0.9973262032085561, 'f1-score': 0.9946666666666667, 'support': 374.0}, '1': {'precision': 0.9933333333333333, 'recall': 0.9802631578947368, 'f1-score': 0.9867549668874173, 'support': 152.0}, 'accuracy': 0.9923954372623575, 'macro avg': {'precision': 0.992677304964539, 'recall': 0.9887946805516465, 'f1-score': 0.990710816777042, 'support': 526.0}, 'weighted avg': {'precision': 0.9924004260712455, 'recall': 0.9923954372623575, 'f1-score': 0.9923803960080243, 'support': 526.0}}" of type <class 'dict'> for key "eval/classification_report" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a v

Trainer is attempting to log a value of "[[373, 1], [3, 149]]" of type <class 'list'> for key "eval/confusion_matrix" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'0': {'precision': 0.9920212765957447, 'recall': 0.9973262032085561, 'f1-score': 0.9946666666666667, 'support': 374.0}, '1': {'precision': 0.9933333333333333, 'recall': 0.9802631578947368, 'f1-score': 0.9867549668874173, 'support': 152.0}, 'accuracy': 0.9923954372623575, 'macro avg': {'precision': 0.992677304964539, 'recall': 0.9887946805516465, 'f1-score': 0.990710816777042, 'support': 526.0}, 'weighted avg': {'precision': 0.9924004260712455, 'recall': 0.9923954372623575, 'f1-score': 0.9923803960080243, 'support': 526.0}}" of type <class 'dict'> for key "eval/classification_report" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.


{'eval_loss': 0.04746996983885765,
 'eval_accuracy': 0.9923954372623575,
 'eval_precision': 0.992677304964539,
 'eval_recall': 0.9887946805516465,
 'eval_f1': 0.990710816777042,
 'eval_mcc': 0.981464305810839,
 'eval_balanced_accuracy': 0.9887946805516465,
 'eval_cohen_kappa': 0.9814219616430615,
 'eval_jaccard': 0.9816230647873649,
 'eval_hamming_loss': 0.0076045627376425855,
 'eval_auc_roc': 0.9887946805516464,
 'eval_auc_pr': 0.9794314922286704,
 'eval_confusion_matrix': [[373, 1], [3, 149]],
 'eval_classification_report': {'0': {'precision': 0.9920212765957447,
   'recall': 0.9973262032085561,
   'f1-score': 0.9946666666666667,
   'support': 374.0},
  '1': {'precision': 0.9933333333333333,
   'recall': 0.9802631578947368,
   'f1-score': 0.9867549668874173,
   'support': 152.0},
  'accuracy': 0.9923954372623575,
  'macro avg': {'precision': 0.992677304964539,
   'recall': 0.9887946805516465,
   'f1-score': 0.990710816777042,
   'support': 526.0},
  'weighted avg': {'precision': 0.99

In [10]:
# # DistilBERT classifier model
# model = DistilBERTClassifier(
#     model_name="distilbert-base-uncased",
#     num_labels=len(class_mapping),
# )

In [11]:
# model.train(
#     X_train.tolist(),
#     y_train.tolist(),
#     epochs=3,
#     batch_size=16,
#     learning_rate=2e-5,
#     weight_decay=0.01,
#     output_dir="../scratch/models/distilbert_degendered"
# )

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/home/hice1/mwesley32/NLP-Letters-V2/src/models.py:185: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  self.trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy,Cohen Kappa,Jaccard,Hamming Loss,Auc Roc,Auc Pr,Confusion Matrix,Classification Report,Runtime,Samples Per Second,Steps Per Second
1,No log,0.042469,0.992395,0.992677,0.988795,0.990711,0.981464,0.988795,0.981422,0.981623,0.007605,0.988795,0.979431,"[[373, 1], [3, 149]]","{'0': {'precision': 0.9920212765957447, 'recall': 0.9973262032085561, 'f1-score': 0.9946666666666667, 'support': 374.0}, '1': {'precision': 0.9933333333333333, 'recall': 0.9802631578947368, 'f1-score': 0.9867549668874173, 'support': 152.0}, 'accuracy': 0.9923954372623575, 'macro avg': {'precision': 0.992677304964539, 'recall': 0.9887946805516465, 'f1-score': 0.990710816777042, 'support': 526.0}, 'weighted avg': {'precision': 0.9924004260712455, 'recall': 0.9923954372623575, 'f1-score': 0.9923803960080243, 'support': 526.0}}",3.106200,169.337000,10.624000
2,No log,0.026817,0.994297,0.996021,0.990132,0.993019,0.986135,0.990132,0.986039,0.986153,0.005703,0.990132,0.985967,"[[374, 0], [3, 149]]","{'0': {'precision': 0.9920424403183024, 'recall': 1.0, 'f1-score': 0.996005326231691, 'support': 374.0}, '1': {'precision': 1.0, 'recall': 0.9802631578947368, 'f1-score': 0.9900332225913622, 'support': 152.0}, 'accuracy': 0.9942965779467681, 'macro avg': {'precision': 0.9960212201591512, 'recall': 0.9901315789473684, 'f1-score': 0.9930192744115266, 'support': 526.0}, 'weighted avg': {'precision': 0.9943419632681466, 'recall': 0.9942965779467681, 'f1-score': 0.9942795472329649, 'support': 526.0}}",3.107400,169.275000,10.620000
3,No log,0.034961,0.994297,0.996021,0.990132,0.993019,0.986135,0.990132,0.986039,0.986153,0.005703,0.990132,0.985967,"[[374, 0], [3, 149]]","{'0': {'precision': 0.9920424403183024, 'recall': 1.0, 'f1-score': 0.996005326231691, 'support': 374.0}, '1': {'precision': 1.0, 'recall': 0.9802631578947368, 'f1-score': 0.9900332225913622, 'support': 152.0}, 'accuracy': 0.9942965779467681, 'macro avg': {'precision': 0.9960212201591512, 'recall': 0.9901315789473684, 'f1-score': 0.9930192744115266, 'support': 526.0}, 'weighted avg': {'precision': 0.9943419632681466, 'recall': 0.9942965779467681, 'f1-score': 0.9942795472329649, 'support': 526.0}}",3.109300,169.168000,10.613000


Trainer is attempting to log a value of "[[373, 1], [3, 149]]" of type <class 'list'> for key "eval/confusion_matrix" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'0': {'precision': 0.9920212765957447, 'recall': 0.9973262032085561, 'f1-score': 0.9946666666666667, 'support': 374.0}, '1': {'precision': 0.9933333333333333, 'recall': 0.9802631578947368, 'f1-score': 0.9867549668874173, 'support': 152.0}, 'accuracy': 0.9923954372623575, 'macro avg': {'precision': 0.992677304964539, 'recall': 0.9887946805516465, 'f1-score': 0.990710816777042, 'support': 526.0}, 'weighted avg': {'precision': 0.9924004260712455, 'recall': 0.9923954372623575, 'f1-score': 0.9923803960080243, 'support': 526.0}}" of type <class 'dict'> for key "eval/classification_report" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a v

Trainer is attempting to log a value of "[[374, 0], [3, 149]]" of type <class 'list'> for key "eval/confusion_matrix" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.
Trainer is attempting to log a value of "{'0': {'precision': 0.9920424403183024, 'recall': 1.0, 'f1-score': 0.996005326231691, 'support': 374.0}, '1': {'precision': 1.0, 'recall': 0.9802631578947368, 'f1-score': 0.9900332225913622, 'support': 152.0}, 'accuracy': 0.9942965779467681, 'macro avg': {'precision': 0.9960212201591512, 'recall': 0.9901315789473684, 'f1-score': 0.9930192744115266, 'support': 526.0}, 'weighted avg': {'precision': 0.9943419632681466, 'recall': 0.9942965779467681, 'f1-score': 0.9942795472329649, 'support': 526.0}}" of type <class 'dict'> for key "eval/classification_report" as a scalar. This invocation of Tensorboard's writer.add_scalar() is incorrect so we dropped this attribute.


{'eval_loss': 0.026817413046956062,
 'eval_accuracy': 0.9942965779467681,
 'eval_precision': 0.9960212201591512,
 'eval_recall': 0.9901315789473684,
 'eval_f1': 0.9930192744115266,
 'eval_mcc': 0.9861352114755968,
 'eval_balanced_accuracy': 0.9901315789473684,
 'eval_cohen_kappa': 0.986039104662479,
 'eval_jaccard': 0.9861527991065195,
 'eval_hamming_loss': 0.005703422053231939,
 'eval_auc_roc': 0.9901315789473684,
 'eval_auc_pr': 0.9859665799479688,
 'eval_confusion_matrix': [[374, 0], [3, 149]],
 'eval_classification_report': {'0': {'precision': 0.9920424403183024,
   'recall': 1.0,
   'f1-score': 0.996005326231691,
   'support': 374.0},
  '1': {'precision': 1.0,
   'recall': 0.9802631578947368,
   'f1-score': 0.9900332225913622,
   'support': 152.0},
  'accuracy': 0.9942965779467681,
  'macro avg': {'precision': 0.9960212201591512,
   'recall': 0.9901315789473684,
   'f1-score': 0.9930192744115266,
   'support': 526.0},
  'weighted avg': {'precision': 0.9943419632681466,
   'recall'

In [14]:
metrics = model.test(X_test.tolist()[0:5], y_test.tolist()[0:5])

print("Evaluation Metrics:", metrics)

Parameter 'function'=<bound method RoBERTaClassifier._tokenize of <src.models.RoBERTaClassifier object at 0x15543baaf700>> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:708: RuntimeWarning: invalid value encountered in true_divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/l

Evaluation Metrics: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'mcc': 0.0, 'balanced_accuracy': 1.0, 'cohen_kappa': nan, 'jaccard': 1.0, 'hamming_loss': 0.0, 'auc_roc': -1, 'auc_pr': -1, 'confusion_matrix': [[5]], 'classification_report': {'0': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 5.0}, 'accuracy': 1.0, 'macro avg': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 5.0}, 'weighted avg': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 5.0}}}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:386: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:708: RuntimeWarning: invalid value encountered in true_divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
/usr/l

In [ ]:
# basic_pipeline


In [ ]:
df["advanced_degendered"] = advanced_pipeline.fit_transform(df[TEXT_COLUMN].tolist())


In [13]:
df["advanced_degendered"][0:5].tolist()

["FIRST_NAME LAST_NAME  * LAST_NAME waived their right to review this letter  * LAST_NAME received a clinical score of outstanding   a theylf board score of satisfactory   and an overall core * LAST_NAME received these comments on their core * \x94 attending e FIRST_NAME performed well on the rotation   specific strengths included actively seeking opportunities for patient care   using strong communication strategies to engender trust with patients   asking for continuous feedback and seeking opportunities to educate team   opportunities infections in hiv    *   attending e it was a pleasure working with FIRST_NAME on wards  * FIRST_NAME's a team player   gets along well with other staff and patients and family  * FIRST_NAME theylped with team work and was active member of the team  * FIRST_NAME has confidence in themself and enjoys the profession they is about to practice   they has a great attitude   that makes working with them to be fun  * \x94 resident in summary   FIRST_NAME comp

In [26]:
model.test(df["advanced_degendered"][0:5].tolist(), df["label"][0:5])

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'accuracy': 0.6, 'precision': 0.3, 'recall': 0.5, 'f1': 0.375}

In [ ]:
# Apply both pipelines to the text column
df["basic_degendered"] = basic_pipeline.transform(df[TEXT_COLUMN].tolist())
df["advanced_degendered"] = advanced_pipeline.transform(df[TEXT_COLUMN].tolist())

print(df[[TEXT_COLUMN, "basic_degendered", "advanced_degendered"]].head())

In [15]:
df["s1_s2"][0]

"FIRST_NAME LAST_NAME  * LAST_NAME waived his right to review this letter  * LAST_NAME received a clinical score of outstanding   a shelf board score of satisfactory   and an overall core * LAST_NAME received these comments on his core * \x94 attending e FIRST_NAME performed well on the rotation   specific strengths included actively seeking opportunities for patient care   using strong communication strategies to engender trust with patients   asking for continuous feedback and seeking opportunities to educate team   opportunities infections in hiv    *   attending e it was a pleasure working with FIRST_NAME on wards  * FIRST_NAME's a team player   gets along well with other staff and patients and family  * FIRST_NAME helped with team work and was active member of the team  * FIRST_NAME has confidence in himself and enjoys the profession he is about to practice   he has a great attitude   that makes working with him to be fun  * \x94 resident in summary   FIRST_NAME completed internal

In [21]:
df["advanced_degendered"][0:5]

"FIRST_NAME LAST_NAME  * LAST_NAME waived their right to review this letter  * LAST_NAME received a clinical score of outstanding   a theylf board score of satisfactory   and an overall core * LAST_NAME received these comments on their core * \x94 attending e FIRST_NAME performed well on the rotation   specific strengths included actively seeking opportunities for patient care   using strong communication strategies to engender trust with patients   asking for continuous feedback and seeking opportunities to educate team   opportunities infections in hiv    *   attending e it was a pleasure working with FIRST_NAME on wards  * FIRST_NAME's a team player   gets along well with other staff and patients and family  * FIRST_NAME theylped with team work and was active member of the team  * FIRST_NAME has confidence in themself and enjoys the profession they is about to practice   they has a great attitude   that makes working with them to be fun  * \x94 resident in summary   FIRST_NAME compl

Allocated: 11.82 GB
Cached: 11.86 GB


3138